# Dimensionnement des systèmes

Ce notebook interactif vous permet d'explorer les concepts de dimensionnement des systèmes en Sciences Industrielles de l'Ingénieur. Vous pourrez modifier les paramètres et observer les résultats en temps réel.

## Introduction

Le dimensionnement d'un système consiste à déterminer les caractéristiques physiques et techniques nécessaires pour qu'il réponde aux exigences fonctionnelles. Cette étape est cruciale dans le processus de conception.

In [1]:
# Importation des bibliothèques nécessaires
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from ipywidgets import interact, FloatSlider, IntSlider

## Dimensionnement d'un moteur électrique

Nous allons étudier le dimensionnement d'un moteur électrique en fonction de différents paramètres comme la charge, la vitesse et le couple.

In [2]:
def calculer_puissance(couple, vitesse):
    """Calcule la puissance mécanique d'un moteur.
    
    Args:
        couple: Couple en N.m
        vitesse: Vitesse de rotation en rad/s
        
    Returns:
        Puissance en Watts
    """
    return couple * vitesse

def calculer_rendement(puissance_mecanique, puissance_electrique):
    """Calcule le rendement d'un moteur.
    
    Args:
        puissance_mecanique: Puissance mécanique en Watts
        puissance_electrique: Puissance électrique en Watts
        
    Returns:
        Rendement (entre 0 et 1)
    """
    return puissance_mecanique / puissance_electrique if puissance_electrique > 0 else 0

In [4]:
# Visualisation de la puissance en fonction du couple et de la vitesse
@interact(couple=FloatSlider(min=0.1, max=10, step=0.1, value=2, description='Couple (N.m)'),
          vitesse=FloatSlider(min=1, max=100, step=1, value=50, description='Vitesse (rad/s)'))
def plot_puissance(couple, vitesse):
    puissance = calculer_puissance(couple, vitesse)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Création d'une plage de valeurs pour le couple et la vitesse
    couples = np.linspace(0.1, couple*1.5, 100)
    vitesses = np.linspace(1, vitesse*1.5, 100)
    
    # Création d'une grille 2D pour les couples et vitesses
    C, V = np.meshgrid(couples, vitesses)
    P = C * V  # Puissance pour chaque combinaison de couple et vitesse
    
    # Tracé du contour de puissance
    contour = ax.contourf(C, V, P, 20, cmap='viridis')
    plt.colorbar(contour, label='Puissance (W)')
    
    # Marquage du point actuel
    ax.plot(couple, vitesse, 'ro', markersize=10, label=f'Point actuel: {puissance:.2f} W')
    
    ax.set_xlabel('Couple (N.m)')
    ax.set_ylabel('Vitesse (rad/s)')
    ax.set_title('Puissance mécanique en fonction du couple et de la vitesse')
    ax.legend()
    ax.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Puissance mécanique calculée: {puissance:.2f} W")

interactive(children=(FloatSlider(value=2.0, description='Couple (N.m)', max=10.0, min=0.1), FloatSlider(value…

## Étude du rendement

Le rendement est un facteur crucial dans le dimensionnement d'un moteur. Il représente le rapport entre la puissance mécanique utile et la puissance électrique consommée.

In [5]:
@interact(puissance_mecanique=FloatSlider(min=10, max=1000, step=10, value=200, description='P. mécanique (W)'),
          rendement=FloatSlider(min=0.1, max=1.0, step=0.05, value=0.8, description='Rendement'))
def plot_rendement(puissance_mecanique, rendement):
    puissance_electrique = puissance_mecanique / rendement
    pertes = puissance_electrique - puissance_mecanique
    
    # Création du graphique
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Données pour le graphique en camembert
    labels = ['Puissance mécanique utile', 'Pertes']
    sizes = [puissance_mecanique, pertes]
    colors = ['#66b3ff', '#ff9999']
    
    ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
    ax.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle
    
    plt.title(f'Répartition de la puissance électrique (rendement = {rendement:.2f})')
    plt.show()
    
    print(f"Puissance électrique nécessaire: {puissance_electrique:.2f} W")
    print(f"Pertes: {pertes:.2f} W")

interactive(children=(FloatSlider(value=200.0, description='P. mécanique (W)', max=1000.0, min=10.0, step=10.0…

## Dimensionnement en fonction du cycle de fonctionnement

Dans de nombreuses applications, le moteur ne fonctionne pas à charge constante. Il est important de considérer le cycle de fonctionnement pour un dimensionnement optimal.

In [6]:
@interact(charge_max=FloatSlider(min=100, max=1000, step=50, value=500, description='Charge max (W)'),
          duree_charge_max=IntSlider(min=1, max=60, step=1, value=10, description='Durée à charge max (s)'),
          charge_nominale=FloatSlider(min=50, max=500, step=25, value=200, description='Charge nominale (W)'),
          duree_nominale=IntSlider(min=10, max=120, step=5, value=30, description='Durée nominale (s)'))
def plot_cycle(charge_max, duree_charge_max, charge_nominale, duree_nominale):
    # Création du cycle de fonctionnement
    temps = np.array([0, duree_charge_max, duree_charge_max, duree_charge_max + duree_nominale, 
                      duree_charge_max + duree_nominale])
    charges = np.array([0, charge_max, charge_nominale, charge_nominale, 0])
    
    # Calcul de la charge RMS (Root Mean Square)
    # Pour simplifier, on utilise une approximation discrète
    temps_discret = np.linspace(0, temps[-1], 1000)
    charges_discret = np.interp(temps_discret, temps, charges)
    charge_rms = np.sqrt(np.mean(charges_discret**2))
    
    # Calcul de l'énergie totale
    energie = np.trapz(charges_discret, temps_discret) / 3600  # en Wh
    
    # Création du graphique
    fig, ax = plt.subplots(figsize=(12, 6))
    
    ax.plot(temps, charges, 'b-', linewidth=2, label='Cycle de charge')
    ax.axhline(y=charge_rms, color='r', linestyle='--', label=f'Charge RMS: {charge_rms:.2f} W')
    
    ax.fill_between(temps, charges, alpha=0.3, color='blue')
    
    ax.set_xlabel('Temps (s)')
    ax.set_ylabel('Puissance (W)')
    ax.set_title('Cycle de fonctionnement du moteur')
    ax.grid(True)
    ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    print(f"Charge RMS: {charge_rms:.2f} W")
    print(f"Énergie totale consommée: {energie:.3f} Wh")
    print(f"Recommandation: Dimensionner le moteur pour une puissance d'au moins {charge_rms*1.2:.2f} W")

interactive(children=(FloatSlider(value=500.0, description='Charge max (W)', max=1000.0, min=100.0, step=50.0)…

## Conclusion

Le dimensionnement d'un système est une étape cruciale qui nécessite de prendre en compte de nombreux facteurs :

1. Les caractéristiques de la charge (couple, vitesse, puissance)
2. Le rendement du système
3. Le cycle de fonctionnement
4. Les contraintes thermiques
5. Les contraintes mécaniques

Ce notebook vous a permis d'explorer interactivement certains de ces aspects. Pour un dimensionnement précis, il est recommandé de :

- Caractériser précisément le cycle de fonctionnement
- Prendre en compte les pics de charge transitoires
- Considérer les conditions environnementales (température, humidité)
- Prévoir une marge de sécurité (généralement 20-30%)

Ces principes s'appliquent à de nombreux systèmes au-delà des moteurs électriques : actionneurs hydrauliques, systèmes thermiques, structures mécaniques, etc.